In [7]:
import os
import sys
import json
import numpy as np
import pandas as pd
import gfootball.env as football_env

# Add src/metrics to the path so we can import calculator.py directly,
# regardless of where the Jupyter kernel's working directory ends up.
PROJECT_ROOT = "/home/praneeth/dissertation/context-aware-magail-grf"
METRICS_DIR = os.path.join(PROJECT_ROOT, "src", "metrics")
sys.path.insert(0, METRICS_DIR)

from calculator import (
    compute_sap, compute_sbf, compute_mecha, compute_cv,
    extract_sprint_sticky_bits, extract_team_positions, extract_possession,
    SPRINT_STICKY_INDEX, GRF_PITCH_AREA,
)

print("calculator.py imported successfully.")
print("SPRINT_STICKY_INDEX =", SPRINT_STICKY_INDEX)

ModuleNotFoundError: No module named 'calculator'

In [ ]:
def make_5v5_env(render=False):
    env = football_env.create_environment(
        env_name='5_vs_5',
        representation='raw',
        number_of_left_players_agent_controls=4,
        number_of_right_players_agent_controls=0,
        render=render,
    )
    return env

In [ ]:
def run_random_episode(env, max_steps=3000, half_time_step=1500, n_players=5):
    obs = env.reset()

    sticky_sprint_log = []
    position_log = []
    possession_log = []

    step = 0
    done = False

    while not done and step < max_steps:
        # Log state BEFORE stepping forward
        sticky_sprint_log.append(extract_sprint_sticky_bits(obs))
        position_log.append(extract_team_positions(obs, n_players=n_players))
        possession_log.append(extract_possession(obs))

        actions = env.action_space.sample()
        obs, reward, done, info = env.step(actions)

        step += 1

    # ── Ground-truth score from the engine (NOT reward-sign inference) ──
    final_score = obs[0]['score']  # [left, right]
    left_score, right_score = int(final_score[0]), int(final_score[1])

    sticky_sprint_array = np.array(sticky_sprint_log)   # (n_steps, n_agents)
    position_array = np.array(position_log)             # (n_steps, n_players, 2)
    possession_array = np.array(possession_log)          # (n_steps,)

    half_idx = min(half_time_step, len(sticky_sprint_array))

    # ── Possession diagnostic -- MECHA is possession-conditional, so a low
    #    possession fraction explains a low/zero MECHA independently of
    #    formation quality. Always log this alongside MECHA. ──
    possession_fraction = float(np.mean(possession_array)) if len(possession_array) > 0 else 0.0

    episode_data = {
        'n_steps': step,
        'left_score': left_score,
        'right_score': right_score,
        'win': int(left_score > right_score),
        'draw': int(left_score == right_score),
        'loss': int(left_score < right_score),

        # Full-match metrics
        'sap': compute_sap(sticky_sprint_array),
        'sbf': compute_sbf(sticky_sprint_array),
        'mecha': compute_mecha(position_array, possession_array),
        'cv': compute_cv(position_array),
        'possession_fraction': possession_fraction,

        # First-half / second-half split
        'sap_1st_half': compute_sap(sticky_sprint_array[:half_idx]),
        'sap_2nd_half': compute_sap(sticky_sprint_array[half_idx:]),
        'sbf_1st_half': compute_sbf(sticky_sprint_array[:half_idx]),
        'sbf_2nd_half': compute_sbf(sticky_sprint_array[half_idx:]),
        'mecha_1st_half': compute_mecha(position_array[:half_idx], possession_array[:half_idx]),
        'mecha_2nd_half': compute_mecha(position_array[half_idx:], possession_array[half_idx:]),
        'cv_1st_half': compute_cv(position_array[:half_idx]),
        'cv_2nd_half': compute_cv(position_array[half_idx:]),
    }
    return episode_data

In [ ]:
def run_random_baseline(n_episodes=500, max_steps=3000, half_time_step=1500, seed=42):
    np.random.seed(seed)
    env = make_5v5_env(render=False)

    results = []
    print(f"Running {n_episodes} random-policy episodes (3000 steps, halftime at {half_time_step})...")
    print("-" * 60)

    for ep in range(n_episodes):
        ep_data = run_random_episode(env, max_steps=max_steps, half_time_step=half_time_step)
        ep_data['episode'] = ep
        results.append(ep_data)

        if (ep + 1) % 10 == 0:
            recent = results[-10:]
            print(f"  Ep {ep+1:3d}/{n_episodes} | "
                  f"SAP: {np.mean([r['sap'] for r in recent]):.2f}% | "
                  f"MECHA: {np.mean([r['mecha'] for r in recent]):.4f} | "
                  f"Win: {np.mean([r['win'] for r in recent]):.2f}")

    env.close()
    return results

In [ ]:
results = run_random_baseline(n_episodes=500, max_steps=3000, half_time_step=1500, seed=42)

df = pd.DataFrame(results)

output_dir = os.path.abspath("../../results")
os.makedirs(output_dir, exist_ok=True)

csv_path = os.path.join(output_dir, "random_baseline_500ep.csv")
df.to_csv(csv_path, index=False)

metrics = ['sap', 'sbf', 'mecha', 'cv','possession_fraction',
           'sap_1st_half', 'sap_2nd_half',
           'mecha_1st_half', 'mecha_2nd_half']

summary = {
    m: {'mean': float(df[m].mean()), 'std': float(df[m].std())}
    for m in metrics
}
summary['win_rate'] = float(df['win'].mean())
summary['n_episodes'] = len(df)

json_path = os.path.join(output_dir, "random_baseline_summary.json")
with open(json_path, 'w') as f:
    json.dump(summary, f, indent=4)

print("\n" + "=" * 60)
print("RANDOM POLICY BASELINE -- NOISE FLOOR (500 episodes)")
print("=" * 60)
for m in metrics:
    print(f"  {m:16s}: {summary[m]['mean']:.4f} ± {summary[m]['std']:.4f}")
print(f"  {'win_rate':16s}: {summary['win_rate']*100:.1f}%")
print(f"\nSaved: {csv_path}")
print(f"Saved: {json_path}")